# Lab 8: MCP & Context Engineering — Remote Servers + Shaping What the Model Sees

**Difficulty: Advanced | ~45 min | Requires Lab 7 (and Lab 5)**

This notebook connects an agent to a **real external MCP server** (Coinfuty on mcp.so — live crypto-futures data, no API key) and to a **self-hosted server** you spawn from the same folder, then **measures the context budget** of those connections and shrinks it with three levers: *prune* tools, *author* lean descriptions, *shape* results. Read the companion `lab-mcp-context-engineering.md` for the full narrative; this notebook is Section 10's build.

**Cost:** 4 OpenRouter calls total on the free model. Everything else (tool discovery and direct tool calls) is free.

In [1]:
# One command installs all required modules (versions pinned for reproducibility).
!pip install langchain==1.3.15 langchain-core==1.5.4 langchain-openai==1.4.3 langgraph==1.2.11 langchain-mcp-adapters==0.3.2 mcp==1.29.0 python-dotenv==1.2.2


[notice] A new release of pip available: 22.3 -> 26.2.1
[notice] To update, run: pip install --upgrade pip


## Step 2 — Imports, the key, and a measuring tool

`.env` holds your OpenRouter key (Labs 5–7). `model()` is a factory returning the free Nemotron model. `UsageCapture` is the lab's instrument: it is a LangChain callback that records `prompt_tokens` — the number of **input tokens** every LLM call carried — so you can *see* the context budget instead of guessing at it. `Path` and the socket/subprocess imports are used by the server-lifecycle cell next.

In [2]:
import asyncio
import os
import socket
import subprocess
import sys
import time
from pathlib import Path

from dotenv import load_dotenv
from langchain.agents import create_agent
from langchain_core.callbacks import BaseCallbackHandler
from langchain_mcp_adapters.client import MultiServerMCPClient
from langchain_openai import ChatOpenAI

load_dotenv()


def model():
    return ChatOpenAI(
        base_url="https://openrouter.ai/api/v1",
        api_key=os.environ["OPENROUTER_API_KEY"],
        model="nvidia/nemotron-3-super-120b-a12b:free",
        temperature=0,
    )


class UsageCapture(BaseCallbackHandler):
    """Records prompt_tokens for every LLM call in a run."""

    def __init__(self):
        self.calls = []

    def on_llm_end(self, response, **kwargs):
        usage = (response.llm_output or {}).get("token_usage", {})
        self.calls.append(usage.get("prompt_tokens", 0))

## Step 3 — Host your own server

`mcp_ops_server.py` binds `127.0.0.1:8788` and speaks Streamable HTTP — the same transport a hosted server would use, just on your machine. The cell is defensive on purpose: if the port already answers it reuses whatever is running there (so re-running the notebook never stacks orphaned processes), and it only spawns a new process when nothing is listening. A short retry loop waits for the port to accept connections. Server errors go to `mcp_ops_server.log` instead of the notebook so a crash is diagnosable.

In [3]:
PORT = 8788
BASE = f"http://127.0.0.1:{PORT}/mcp"


def port_up(port, timeout=20):
    deadline = time.time() + timeout
    while time.time() < deadline:
        try:
            sock = socket.create_connection(("127.0.0.1", port), timeout=1)
            sock.close()
            return True
        except OSError:
            time.sleep(0.3)
    return False


server = None
if not port_up(PORT, timeout=1):
    logfile = open("mcp_ops_server.log", "w")
    server = subprocess.Popen(
        [sys.executable, str(Path("mcp_ops_server.py").resolve()), str(PORT)],
        stdout=subprocess.DEVNULL,
        stderr=logfile,
    )

print("server ready:", port_up(PORT))
print("endpoint:", BASE)

server ready: True
endpoint: http://127.0.0.1:8788/mcp


## Step 4 — Connect two remote servers

The same `MultiServerMCPClient` you used over stdio in Lab 7 — the only difference is the connection dict: `"transport": "http"` plus a URL. One entry points at a **real external server** (Coinfuty, HTTPS, on mcp.so) and one at **your own server** (HTTP, localhost). Everything else — `tools/list`, `tools/call`, the adapter, the agent — is identical. That is the whole "remote" trick.

In [4]:
client = MultiServerMCPClient({
    "coinfuty": {"transport": "http", "url": "https://mcp.coinfuty.com/api/mcp"},
    "ops":      {"transport": "http", "url": "http://127.0.0.1:8788/mcp"},
})
tools = await client.get_tools()

external = [t for t in tools if t.name.startswith(("get_", "list_"))]
local = [t for t in tools if t.name.startswith("digest_")]
print("external tools:", [t.name for t in external])
print("local tools:   ", [t.name for t in local])

external tools: ['get_market_overview', 'get_coins_markets', 'get_coin_summary', 'get_price_history', 'get_funding_rates', 'get_exchange_breakdown', 'list_exchanges']
local tools:    ['digest_snapshot', 'digest_logs', 'digest_highlights']


## Step 5 — Take the context ledger

Before any model call, serialize the **fixed cost**: each tool's JSON schema. Every character of that schema is sent on *every* request while the tool is bound — used or not. External schemas are someone else's design (you can only take them or leave them); your server's tools are yours to author. Rough rule of thumb: about one token per 4 characters of schema.

In [5]:
ext_total = 0
for t in external:
    print(f"  {t.name}: schema {len(str(t.args))} chars")
    ext_total += len(str(t.args))
print("EXTERNAL schema total:", ext_total, "chars -> paid on every call while bound")
print()
for t in local:
    print(f"  {t.name}: schema {len(str(t.args))} chars | description {len(t.description or '')} chars")

  get_market_overview: schema 2 chars
  get_coins_markets: schema 568 chars
  get_coin_summary: schema 70 chars
  get_price_history: schema 406 chars
  get_funding_rates: schema 70 chars
  get_exchange_breakdown: schema 285 chars
  list_exchanges: schema 2 chars
EXTERNAL schema total: 1403 chars -> paid on every call while bound

  digest_snapshot: schema 73 chars | description 181 chars
  digest_logs: schema 111 chars | description 218 chars
  digest_highlights: schema 47 chars | description 264 chars


## Step 6 — A/B 1: prune the external tools

Same question, two agents: one bound to **all 7** external tools, one bound to only the **2** the question needs (`get_funding_rates`, `get_coin_summary`). `run_with_usage` runs the agent and hands back the per-call input tokens; `calls[0]` is the **decision-time context** — what the model saw before it chose a tool. Expect identical answers with a much smaller first call.

In [6]:
async def run_with_usage(agent, question):
    capture = UsageCapture()
    result = await agent.ainvoke(
        {"messages": [("human", question)]}, config={"callbacks": [capture]}
    )
    return capture.calls, str(result["messages"][-1].content)


Q1 = "What is the current funding rate and open interest for BTC futures?"
pruned_names = {"get_funding_rates", "get_coin_summary"}
pruned = [t for t in external if t.name in pruned_names]

calls_all, ans_all = await run_with_usage(create_agent(model=model(), tools=external), Q1)
calls_2, ans_2 = await run_with_usage(create_agent(model=model(), tools=pruned), Q1)

print("ALL 7 tools : first-call tokens =", calls_all[0], "| per-call:", calls_all)
print("PRUNED 2    : first-call tokens =", calls_2[0], "| per-call:", calls_2)
print("answer (all):", ans_all[:70].replace("\n", " "))
print("answer (2)  :", ans_2[:70].replace("\n", " "))

ALL 7 tools : first-call tokens = 1795 | per-call: [1795, 5325]
PRUNED 2    : first-call tokens = 551 | per-call: [551, 4076]
answer (all):  **BTC Futures (as of the latest data)**    | Metric | Value | Details
answer (2)  :   **BTC Futures (as of the latest data)**    | Metric | Value | Notes 


## Step 7 — A/B 2: shape the results on your own server

Now the **variable cost**. Both agents answer *"Read the recent logs for BTC and summarize what happened."* — one bound only to `digest_logs` (the raw firehose, 300 lines ≈ 14 KB by default), one only to `digest_highlights` (the *same* events, compressed server-side to ~160 chars). First measure the result sizes directly (free — no model involved), then run the agents and watch the **second** call: that is how much context the tool result dragged into the model.

In [7]:
logs_fat = [t for t in local if t.name == "digest_logs"][0]
logs_lean = [t for t in local if t.name == "digest_highlights"][0]
fat_result = await logs_fat.ainvoke({"coins": "BTC"})
lean_result = await logs_lean.ainvoke({"coins": "BTC"})
print("fat result chars: ", len(str(fat_result)))
print("lean result chars:", len(str(lean_result)))

Q2 = "Read the recent logs for BTC and summarize what happened."
calls_fat, ans_fat = await run_with_usage(create_agent(model=model(), tools=[logs_fat]), Q2)
calls_lean, ans_lean = await run_with_usage(create_agent(model=model(), tools=[logs_lean]), Q2)

print("FAT  run per-call tokens:", calls_fat)
print("LEAN run per-call tokens:", calls_lean)
print("answer (fat) :", ans_fat[:70].replace("\n", " "))
print("answer (lean):", ans_lean[:70].replace("\n", " "))

fat result chars:  14214
lean result chars: 163


FAT  run per-call tokens: [358, 6490]
LEAN run per-call tokens: [331, 416]
answer (fat) :   **Recent BTC Log Summary (last ~300 lines)**    | Log Level | Approx
answer (lean):   **Recent BTC Log Summary**  - **Error count:** 9   - **Warning count


## Step 8 — Close the ledger

The ratios are the lesson. Two runs, four numbers: pruning cut **decision-time** context, result shaping cut the **post-tool** context — and both answers stayed correct. Read each row and name which lever produced it (prune vs shape) and which server you control in that row (external vs your own).

In [8]:
fat_ctx = calls_fat[1] if len(calls_fat) > 1 else 0
lean_ctx = calls_lean[1] if len(calls_lean) > 1 else 0
print(f"prune: decision tokens   {calls_all[0]:>6} -> {calls_2[0]:>6}   ({(1 - calls_2[0] / calls_all[0]):.0%} less)")
print(f"shape: result chars      {len(str(fat_result)):>6} -> {len(str(lean_result)):>6}   ({(1 - len(str(lean_result)) / len(str(fat_result))):.0%} less)")
print(f"shape: post-result ctx   {fat_ctx:>6} -> {lean_ctx:>6}   ({(1 - lean_ctx / fat_ctx):.0%} less)")

prune: decision tokens     1795 ->    551   (69% less)
shape: result chars       14214 ->    163   (99% less)
shape: post-result ctx     6490 ->    416   (94% less)


## Step 9 — Shut the server down

Terminate the process this kernel spawned — and only that one. If a server was already running on the port when you started, leave it alone.

In [9]:
if server is not None and server.poll() is None:
    server.terminate()
    print("stopped the server we spawned")
else:
    print("no server process to stop (server already running or already exited)")

stopped the server we spawned
